### [순위](https://school.programmers.co.kr/learn/courses/30/lessons/49191)
- 이전에 풀었던 것을 다시 풀어보기

- ??? 논리는 맞는 것 같은데 일부 케이스를 틀린다

In [1]:
def solution(n, results):
    board = [[False] * (n+1) for _ in range(n+1)]
    
    # 1차 절대 전적 정리
    for win, lose in results:
        board[win][lose] = True
    
    # 2차 상대 전적 정리
    for i in range(1, n+1):
        for j in range(1, n+1):
            for k in range(1, n+1):
                if board[i][j] and board[j][k]:
                    board[i][k] = True
    
    # 카운트
    answer = 0
    for i in range(1, n+1):
        cnt = 0
        for j in range(1, n+1):
            if i == j:
                continue
            elif board[i][j] or board[j][i]:
                cnt += 1
        if cnt == n-1:
            answer += 1
    
    return answer

- 거쳐가는 지점을 가장 상위 반복문에 둬야, 모든 상대전적 관리가 가능해짐 -> 플로이드 워셜

In [ ]:
def solution(n, results):
    board = [[False] * (n+1) for _ in range(n+1)]
    
    # 1차 절대 전적 정리
    for win, lose in results:
        board[win][lose] = True
    
    # 2차 상대 전적 정리
    for k in range(1, n+1):
        for i in range(1, n+1):
            for j in range(1, n+1):
                if board[i][k] and board[k][j]:
                    board[i][j] = True
    
    # 카운트
    answer = 0
    for i in range(1, n+1):
        cnt = 0
        for j in range(1, n+1):
            if i == j:
                continue
            elif board[i][j] or board[j][i]:
                cnt += 1
        if cnt == n-1:
            answer += 1
    
    return answer

### [파괴되지 않은 건물](https://school.programmers.co.kr/learn/courses/30/lessons/92344)

- 효율성 충족 X

In [ ]:
def solution(board, skill):
    for (t, r1, c1, r2, c2, degree) in skill:
        for i in range(r1, r2+1):
            board[i][c1:c2+1] = [x-degree if t == 1 else x+degree for x in board[i][c1:c2+1]]
            
    return sum([sum([1 for y in x if y > 0]) for x in board])

- 누적합을 이용해서 보드의 경계만을 표시한 뒤, 최종적으로 한 번만 계산하는 식으로 진행

In [ ]:
def solution(board, skill):
    N, M = len(board), len(board[0])
    diff = [[0] * (M + 1) for _ in range(N + 1)]
    
    for t, r1, c1, r2, c2, degree in skill:
        v = -degree if t == 1 else degree
        diff[r1][c1] += v
        diff[r1][c2 + 1] -= v
        diff[r2 + 1][c1] -= v
        diff[r2 + 1][c2 + 1] += v
        
    # 행 기준(왼쪽에서 오른쪽) 누적 합
    for i in range(N):
        for j in range(1, M):
            diff[i][j] += diff[i][j - 1]
            
    # 열 기준(위에서 아래) 누적 합
    for j in range(M):
        for i in range(1, N):
            diff[i][j] += diff[i - 1][j]
            
    answer = 0
    for i in range(N):
        for j in range(M):
            if board[i][j] + diff[i][j] > 0:
                answer += 1
                
    return answer

### [자물쇠와 열쇠](https://school.programmers.co.kr/learn/courses/30/lessons/60059)

In [22]:
def get_coords(matrix, target_value):
    return [(r, c) for r in range(len(matrix)) for c in range(len(matrix)) if matrix[r][c] == target_value]

def solution(key, lock):
    m, n = len(key), len(lock)
    
    lock_holes = set(get_coords(lock, 0))
    lock_bumps = set(get_coords(lock, 1))
    key_bumps = get_coords(key, 1)
    
    # 예외 처리: 애초에 자물쇠에 홈이 없다면 이미 열린 상태
    if not lock_holes:
        return True
        
    # Step 1. 하나의 자물쇠 타겟 홈을 고정으로 잡습니다.
    target_hole = list(lock_holes)[0]
    
    # Step 2. 열쇠를 4방향으로 먼저 돌려가며 체크 (회전 먼저!)
    for _ in range(4):
        key_bumps = [(c, m - 1 - r) for r, c in key_bumps] # 90도 회전
        
        # Step 3. 열쇠의 각 돌기를 타겟 홈으로 하나씩 이동시켜 봅니다.
        for kr, kc in key_bumps:
            # 타겟 홈으로 가기 위한 이동 거리(벡터) 계산
            dr = target_hole[0] - kr
            dc = target_hole[1] - kc
            
            # Step 4. 계산된 거리만큼 열쇠의 모든 돌기를 평행 이동시킵니다.
            moved_key = set()
            for r, c in key_bumps:
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n: # 자물쇠 영역 안에 들어온 것만 남김
                    moved_key.add((nr, nc))
            
            # Step 5. 검증 (홈을 다 채웠는가? & 돌기끼리 안 겹치는가?)
            if lock_holes.issubset(moved_key) and lock_bumps.isdisjoint(moved_key):
                return True
                
    return False

### [양과 늑대](https://school.programmers.co.kr/learn/courses/30/lessons/92343)

- 단순한 탐색 문제가 아니라 특정 조건(모두 순회 or 늑대로 인해 더이상 탐색 불가)를 만족할 때까지 반복 탐색하는 문제...

In [ ]:
from collections import deque, defaultdict

def solution(info, edges):
    maps = defaultdict(list)
    for s, e in edges:
        maps[s].append((e, info[e]))
    
    answer = 1
    queue = deque([(0, 1, 0)])
    while queue:
        cur, sheep, wolf = queue.popleft()
        print(f'\n{cur=}, {sheep=}, {wolf=}')
        for nex, animal in maps[cur]:
            print(f'{nex=}, {animal=}')
            if animal == 1: # 늑대
                print(f'Wolf!')
                if sheep > wolf+1:
                    queue.append((nex, sheep, wolf+1))
                else:
                    continue
            elif animal == 0: # 양
                print(f'Sheep! {answer=}, {sheep+1=}')
                queue.append((nex, sheep+1, wolf))
                if sheep+1 > answer:
                    answer = sheep+1
    return answer
                
                    

In [1]:
any([False, True, False])

True

In [3]:
all([1, True, True])

True

In [5]:
True is True

True

In [4]:
1 is True

<>:1: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:1: SyntaxWarning: "is" with a literal. Did you mean "=="?
/tmp/ipykernel_108643/3482963539.py:1: SyntaxWarning: "is" with a literal. Did you mean "=="?
  1 is True


False

- 양방향 탐색과 히스토리를 추가했으나, 여전히 풀지 못함

In [ ]:
from collections import deque, defaultdict

def solution(info, edges):
    maps = defaultdict(list)
    history = info.copy() # 가볼만한 후보 지역 중 양이 있는 곳
    for s, e in edges:
        maps[s].append((e, info[e]))
        maps[e].append((s, -1))
    
    answer = 1
    history[0] = True
    queue = deque([(0, 1, 0, history)])
    while queue:
        cur, sheep, wolf, hist = queue.popleft()
        print(f'\n{cur=}, {hist=}')
        for nex, animal in maps[cur]:
            if animal == -1 or hist[nex] is True: # 가봤던 곳(아무것도 없음)
                print(f'scout -1: {nex=}')
                if not all(hist) : # 아직 추가로 가볼만한 곳(양이 있고, 아직 가지 않은 곳)이 있다면
                    # print(history, answer)
                    queue.append((nex, sheep, wolf, hist))
            elif animal == 1: # 늑대
                print('scout 1')
                if sheep > wolf+1:
                    queue.append((nex, sheep, wolf+1, hist))
                else:
                    continue
            elif animal == 0 and not hist[nex]: # 양
                print('scout 0')
                # print(f'\n{nex=}, {sheep=}, {answer=}')
                hist[nex] = True
                queue.append((nex, sheep+1, wolf, hist))
                if sheep+1 > answer:
                    answer = sheep+1

    return answer
                
                    